In [ ]:
from pathlib import Path
import pandas as pd
from formulation.common import ProblemDataReal

import matplotlib.pyplot as plt

from formulation.bird_adapter import (
    BirdAdapterConfig,
    BirdBackendSolution,
    BirdExportInstance,
    export_bird_instance,
    normalized_result_from_bird_solution,
    routing_solution_json_from_bird_solution,
)
import subprocess
import osmnx as ox
import numpy as np

In [3]:
cache_path = Path("formulation/cache/framingham_problem_data.pkl")
school_name_to_code = pd.read_csv("experiments/data/schools.csv").set_index("name")["id"].to_dict()
problem_data = ProblemDataReal.load_path(cache_path)

out_dir = Path("experiments/outputs/evaluate_bird")
out_dir.mkdir(parents=True, exist_ok=True)


In [4]:
bird_problem = problem_data
config = BirdAdapterConfig(
    cohort="conventional",
    bus_type="C",
    school_dwell_time=10,
    max_time_on_bus=60,
)

instance_path = export_bird_instance(
    bird_problem,
    out_dir / "bird_all_instance.npz",
    config,
)
solution_path = out_dir / "bird_all.npz"

subprocess.run(
    [
        "julia",
        "--project=julia",
        "experiments/solve_bird_backend_julia.jl",
        "--instance",
        str(instance_path),
        "--solution",
        str(solution_path),
        "--method",
        "scenario",
    ],
    check=True,
)

instance = BirdExportInstance.load(instance_path)
solution = BirdBackendSolution.load(solution_path)
normalized = normalized_result_from_bird_solution(instance, solution)
normalized.save(out_dir / "bird_all_normalized.json")

normalized


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2440707
Academic license 2440707 - for non-commercial use only - registered to ri___@mit.edu
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.3.0 25D2128)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Academic license 2440707 - for non-commercial use only - registered to ri___@mit.edu
Optimize a model with 224 rows, 100 columns and 2016 nonzeros (Min)
Model fingerprint: 0x334dc552
Model has 100 linear objective coefficients
Variable types: 0 continuous, 100 integer (100 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+04, 1e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 104604.55822
Presolve removed 87 rows and 2 columns
Presolve time: 0.00s
Presolved: 137 rows, 98 columns, 1213 nonzeros
Variable types: 0 continuous, 98 in

NormalizedRoutingResult(backend='bird', status='OPTIMAL', objective_value=59.0, runtime_seconds=6.664986848831177, buses_used=59, total_distance_km=3229.6949732258036, routes=[NormalizedRoute(bus_id='bird_bus_1', order=0, school_id='CAM', stop_ids=['625 GROVE ST', 'GROVE ST & BONVINI DR ***', 'GROVE ST & BLACKBERRY LN', 'GROVE ST & RALEIGH RD', 'BROOK ST & JEAN ST', '40 WESLEY RD', '27 LINDA AV', 'BROOK ST & SALVI DR', 'WATER ST & LARIVIERE RD', '32 SWANSON RD - W/C', 'GRIFFIN RD & BRADFORD RD', 'NICHOLAS RD @ MILL FALLS ENTRANCE', 'PINEWOOD DR & BRADFORD RD', '73 NICHOLAS RD @ ENTRANCE', 'CENTRAL ST & CENTENNIAL PL', 'CENTRAL ST & JOHNSON ST', 'DANFORTH ST & COTTAGE ST', 'DANFORTH ST & MEADOW ST', 'DANFORTH ST & HIALEAH LN', 'DANFORTH ST & BRIGATI TER', 'OLD CONN PATH & SAXONY TER', 'OLD CON PATH & CHAPPLEWOOD RD', 'SCHOOL ST & MEADOW ST', 'SCHOOL ST & COTTAGE ST', 'SCHOOL ST & QUEENS WY', 'A ST @ WINCH PARK RD', '1585 CONCORD ST @ ENTRANCE', 'ELM ST & CHESTNUT ST- RIGHT SIDE', '78 EL

In [5]:
instance = BirdExportInstance.load(out_dir / "bird_all_instance.npz")
solution = BirdBackendSolution.load(out_dir / "bird_all.npz")

In [6]:
# Bird stores stop ids local to each school-route, so map them back school-by-school.
school_demand_rows = {
    school_idx: [
        row
        for row, idx in zip(instance.demand_rows, instance.demand_school_indices)
        if int(idx) == school_idx
    ]
    for school_idx in range(1, len(instance.schools) + 1)
}

rows = []
for bus_id in sorted(set(solution.assignment_bus_ids.tolist())):
    bus_rows = np.where(solution.assignment_bus_ids == bus_id)[0]
    bus_rows = sorted(bus_rows, key=lambda i: int(solution.assignment_orders[i]))
    if not bus_rows:
        continue

    student_names_for_bus = set()
    total_students = 0
    total_distance_km = 0.0
    arrival_summaries = []

    for row_idx in bus_rows:
        q = int(solution.assignment_orders[row_idx])
        school_idx = int(solution.assignment_school_indices[row_idx])
        school = instance.schools[school_idx - 1]

        start = int(solution.assignment_stop_ptr[row_idx])
        end = int(solution.assignment_stop_ptr[row_idx + 1])
        local_stop_ids = solution.assignment_stop_values[start:end].tolist()

        demand_rows_for_school = school_demand_rows[school_idx]
        route_demand_rows = [demand_rows_for_school[local_stop_id - 1] for local_stop_id in local_stop_ids]

        student_names = [i for s in [row.student_names for row in route_demand_rows] for i in s]
        student_names_for_bus = student_names_for_bus.union(student_names)
        route_students = sum(row.students for row in route_demand_rows)
        total_students += route_students
        total_distance_km += float(solution.assignment_distance_km[row_idx])

        arrival = float(solution.assignment_arrival_times[row_idx])
        slack = float(school.start_time) - arrival
        stop_names = ", ".join(row.stop_name for row in route_demand_rows)

        arrival_summaries.append(
            f"round {q}: {school.name}, {slack:.1f} min before start "
            f"(T={arrival:.1f}), {route_students} students, stops=[{stop_names}]"
        )

    rows.append(
        {
            "bus": f"bird_bus_{bus_id}",
            "rounds_used": len(bus_rows),
            "students": total_students,
            # "student_ids": list(sorted(student_names_for_bus)),
            "distance_km": round(total_distance_km, 2),
            "arrival_vs_school_start": "; ".join(arrival_summaries),
        }
    )

bird_report_df = pd.DataFrame(rows).sort_values("bus").reset_index(drop=True)
print(f"Buses used: {len(bird_report_df)}")
bird_report_df


Buses used: 59


,bus,rounds_used,students,distance_km,arrival_vs_school_start
0,bird_bus_1,2,142,48.72,"round 0: CAMERON, 10.0 min before start (T=450..."
1,bird_bus_10,2,133,55.45,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
2,bird_bus_11,2,135,38.68,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
3,bird_bus_12,3,211,61.33,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
4,bird_bus_13,2,142,41.92,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
5,bird_bus_14,3,196,58.61,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
6,bird_bus_15,2,142,34.56,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
7,bird_bus_16,2,114,51.59,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
8,bird_bus_17,2,142,39.11,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."
9,bird_bus_18,2,101,55.61,"round 0: FRAMINGHAM HIGH SCHOOL, 10.0 min befo..."


In [ ]:
graph = bird_problem.osm_graph

def service_path(u, v):
    edge = bird_problem.service_graph.get_edge_data(u, v, key=0)
    if edge is None:
        raise ValueError(f"missing service edge {u}->{v}")
    return edge["path"]

# Bird stop ids in the solution are local to each school.
school_demand_rows = {
    school_idx: [
        row
        for row, idx in zip(instance.demand_rows, instance.demand_school_indices)
        if int(idx) == school_idx
    ]
    for school_idx in range(1, len(instance.schools) + 1)
}

route_rows = []
for row_idx, bus_id in enumerate(solution.assignment_bus_ids.tolist()):
    school_idx = int(solution.assignment_school_indices[row_idx])
    school = instance.schools[school_idx - 1]

    start = int(solution.assignment_stop_ptr[row_idx])
    end = int(solution.assignment_stop_ptr[row_idx + 1])
    local_stop_ids = solution.assignment_stop_values[start:end].tolist()

    demand_rows_for_school = school_demand_rows[school_idx]
    route_demand_rows = [demand_rows_for_school[stop_id - 1] for stop_id in local_stop_ids]
    stop_nodes = [row.stop_node_id for row in route_demand_rows]

    route_rows.append(
        {
            "bus_id": bus_id,
            "order": int(solution.assignment_orders[row_idx]),
            "school_node": school.node_id,
            "school_name": school.name,
            "stop_nodes": stop_nodes,
            "students": sum(row.students for row in route_demand_rows),
        }
    )

if len(bird_problem.depots) != 1:
    raise ValueError("This plotting cell assumes a single depot.")
depot_node = bird_problem.depots[0].node_id

bus_route_figures = {}
for bus_id in sorted({row["bus_id"] for row in route_rows}):
    bus_rows = sorted(
        [row for row in route_rows if row["bus_id"] == bus_id],
        key=lambda row: row["order"],
    )
    
    print(bus_rows)

    fig, ax = ox.plot_graph(
        graph,
        show=False,
        close=False,
        node_size=0,
        edge_color="#d0d0d0",
        edge_linewidth=0.5,
        bgcolor="white",
        figsize=(10, 10),
    )

    cmap = plt.get_cmap("Dark2", max(1, len(bus_rows)))
    current_node = depot_node
    route_node_ids = [depot_node]

    stop_nodes = sorted({node for row in bus_rows for node in row["stop_nodes"]})
    school_nodes = sorted({row["school_node"] for row in bus_rows})
    school_names = {row["school_node"]: row["school_name"] for row in bus_rows}
    linewidths = [3,2,1,0.5]

    for color_idx, row in enumerate(bus_rows):
        route_color = cmap(color_idx)
        ax.plot([], [], color=route_color, linewidth=3, label=f"Round {row['order']}")

        visit_nodes = [current_node] + row["stop_nodes"] + [row["school_node"]]
        visit_nodes = [
            node for i, node in enumerate(visit_nodes)
            if i == 0 or node != visit_nodes[i - 1]
        ]

        for u, v in zip(visit_nodes, visit_nodes[1:]):
            route_nodes = service_path(u, v)
            if len(route_nodes) >= 2:
                route_node_ids.extend(route_nodes)
                ox.plot_graph_route(
                    graph,
                    route_nodes,
                    ax=ax,
                    show=False,
                    close=False,
                    orig_dest_size=0,
                    route_color=route_color,
                    route_linewidth=linewidths[color_idx % len(linewidths)],
                    route_alpha=0.9,
                )

        current_node = row["school_node"]

    if stop_nodes:
        ax.scatter(
            [graph.nodes[node_id]["x"] for node_id in stop_nodes],
            [graph.nodes[node_id]["y"] for node_id in stop_nodes],
            c="#1f78b4",
            s=35,
            marker="o",
            label="Pickup stops",
            zorder=4,
        )

    if school_nodes:
        ax.scatter(
            [graph.nodes[node_id]["x"] for node_id in school_nodes],
            [graph.nodes[node_id]["y"] for node_id in school_nodes],
            c="#e31a1c",
            s=90,
            marker="s",
            label="Schools",
            zorder=5,
        )

    ax.scatter(
        graph.nodes[depot_node]["x"],
        graph.nodes[depot_node]["y"],
        c="black",
        s=120,
        marker="X",
        label="Depot",
        zorder=6,
    )

    focus_nodes = sorted(set(route_node_ids) | set(stop_nodes) | set(school_nodes) | {depot_node})
    if focus_nodes:
        xs = [graph.nodes[node_id]["x"] for node_id in focus_nodes]
        ys = [graph.nodes[node_id]["y"] for node_id in focus_nodes]
        x_pad = max((max(xs) - min(xs)) * 0.15, 0.005)
        y_pad = max((max(ys) - min(ys)) * 0.15, 0.005)
        ax.set_xlim(min(xs) - x_pad, max(xs) + x_pad)
        ax.set_ylim(min(ys) - y_pad, max(ys) + y_pad)

    total_students = sum(row["students"] for row in bus_rows)
    ax.set_title(f"Bird bus {bus_id}: {total_students} students, {len(bus_rows)} route(s), {[school_name_to_code[school_names[row['school_node']]] for row in bus_rows]}")
    ax.legend(loc="best")

    bus_route_figures[f"bird_bus_{bus_id}"] = fig
    plt.show()


In [8]:
report = routing_solution_json_from_bird_solution(instance, solution)
report.save(out_dir / "bird_solution.json")


PosixPath('experiments/outputs/evaluate_bird/bird_solution.json')